# Annotate samples 

Use `pyannotations` for best ease to annotate samples. Note, the samples here are larger than the intended ones and additionally marked, as easier to annotate in context + may want to make a judgement call whether to treat as labelled or keep for the unsupervised portion.

In [1]:
# re-loads code before cell execution, so if sth changes it will propagate:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, shutil
import logging

import time as tm
import pandas as pd

from PIL import Image
from ipyannotations import images

sys.path.append('..')

import src.geom_utils as sgut

## Set up annnotation

In [3]:
data_path = '../data/xView/to_annotate/'
df = pd.read_csv(f'{data_path}labels.csv')
out_path_ims = '../data/xView/images-0/'
out_path_labels = '../data/xView/labels-0/'

to_annot_names = [ x for x in os.listdir( data_path) if not x.endswith('.csv')]
to_annot = [os.path.join( data_path, name) for name in to_annot_names]
labels = []


In [4]:
widget = images.ClassLabeller(
    options=['road','no road','unlabel','reject']
)

def annotate( annot):
    labels.append( annot)
    try:
        widget.display( to_annot.pop(0))
    except IndexError:
        print('done')
        
widget.on_submit( annotate)

## Annotate interactively


In [5]:
widget.display( to_annot.pop(0))
widget


ClassLabeller(children=(Box(children=(Output(layout=Layout(margin='auto', min_height='50px')),), layout=Layout…

done


## IMPORTANT - save non-rejected image crops & updated file

In [7]:
len(labels[:-2]), len(df)

(492, 492)

In [6]:
df['labels'] = labels
df.loc[ df['labels'] == 'road', 'roads'] = 1
df.loc[ df['labels'] == 'no road', 'roads'] = 0


In [7]:
df.labels.unique()

array(['road', 'unlabel', 'reject', 'no road'], dtype=object)

In [8]:
len( df.loc[ df['labels']=='reject']) / len(df)

0.5101351351351351

In [9]:
len( df.loc[ pd.notnull( df['roads'])]) / len(df)

0.35135135135135137

In [10]:
name_split = df['im_name'].str.split('_').map(lambda x: x[0]).unique()
assert len(name_split)==1
which_tiff = name_split[0]

In [11]:
out_to_ = f'{out_path_labels}for_tiff{which_tiff}.csv'
print(f'Outputting to {out_to_}.')
df.to_csv( out_to_, index= False)

Outputting to ../data/xView/labels-0/for_tiff1132.csv.


In [12]:
df.head(3)

,xs,ys,bbox_np,bbox_context,bbox_in_context,built_area,n_cars,n_bus_trucks,roads,im_name,labels
0,89,3844,"[25, 3780, 153, 3908]","[0, 3665, 256, 3921]","[25, 115, 153, 243]",0.200195,6.0,0.0,1.0,1132_0.png,road
1,93,560,"[29, 496, 157, 624]","[0, 432, 256, 688]","[29, 64, 157, 192]",0.140564,0.0,0.0,NaN,1132_1.png,unlabel
2,93,2097,"[29, 2033, 157, 2161]","[0, 1969, 256, 2225]","[29, 64, 157, 192]",0.000000,0.0,0.0,1.0,1132_2.png,road
